<a href="https://colab.research.google.com/github/Ashwini9713/Digital-forensics-/blob/main/AT7_3_.ipynb" target="_parent"><img src="https://colab.research.google.com/assets/colab-badge.svg" alt="Open In Colab"/></a>

In [2]:
import re
AUTH_LINE_RE = re.compile(
r"(?P<result>Accepted|Failed) password for (?P<user>\S+) from (?P<ip>[\d.]+) port (?P<port>\d+)"
)

def parse_auth_log(lines):
    """Parse raw auth.log lines into structured dicts."""
    entries = []
    for line in lines:
        match = AUTH_LINE_RE.search(line)
        if match:
            entries.append({
                "result": match.group("result"),
                "user": match.group("user"),
                "ip": match.group("ip"),
                "port": int(match.group("port")),
                "raw": line,
            })
    return entries

def flag_suspicious_logins(entries, trusted_ips):
    """Flag successful logins from untrusted IPs, especially for root."""
    flagged = []
    for e in entries:
        if e["result"] == "Accepted" and e["ip"] not in trusted_ips:
            severity = "HIGH" if e["user"] == "root" else "MEDIUM"
            flagged.append({**e, "severity": severity})
    return flagged

### Test Case: Verifying Log Parsing and Suspicious Login Flagging

In [3]:
sample_log_lines = [
    "Accepted password for user1 from 192.168.1.100 port 22",
    "Failed password for user2 from 10.0.0.5 port 22",
    "Accepted password for root from 192.168.1.101 port 22",
    "Accepted password for user3 from 172.16.0.1 port 22",
    "Failed password for root from 1.2.3.4 port 22",
    "Accepted password for user4 from 192.168.1.100 port 22"
]

parsed_entries = parse_auth_log(sample_log_lines)
print("Parsed Entries:")
display(parsed_entries)

Parsed Entries:


[{'result': 'Accepted',
  'user': 'user1',
  'ip': '192.168.1.100',
  'port': 22,
  'raw': 'Accepted password for user1 from 192.168.1.100 port 22'},
 {'result': 'Failed',
  'user': 'user2',
  'ip': '10.0.0.5',
  'port': 22,
  'raw': 'Failed password for user2 from 10.0.0.5 port 22'},
 {'result': 'Accepted',
  'user': 'root',
  'ip': '192.168.1.101',
  'port': 22,
  'raw': 'Accepted password for root from 192.168.1.101 port 22'},
 {'result': 'Accepted',
  'user': 'user3',
  'ip': '172.16.0.1',
  'port': 22,
  'raw': 'Accepted password for user3 from 172.16.0.1 port 22'},
 {'result': 'Failed',
  'user': 'root',
  'ip': '1.2.3.4',
  'port': 22,
  'raw': 'Failed password for root from 1.2.3.4 port 22'},
 {'result': 'Accepted',
  'user': 'user4',
  'ip': '192.168.1.100',
  'port': 22,
  'raw': 'Accepted password for user4 from 192.168.1.100 port 22'}]

In [4]:
trusted_ips = ["192.168.1.100", "192.168.1.101"]

flagged_logins = flag_suspicious_logins(parsed_entries, trusted_ips)
print("Flagged Suspicious Logins:")
display(flagged_logins)

Flagged Suspicious Logins:


[{'result': 'Accepted',
  'user': 'user3',
  'ip': '172.16.0.1',
  'port': 22,
  'raw': 'Accepted password for user3 from 172.16.0.1 port 22',
  'severity': 'MEDIUM'}]